In [14]:
# ---- Set env vars BEFORE importing TensorFlow ----
import os

# # Hide most TensorFlow C++ INFO/WARNING logs
# # 0 = all logs, 1 = no INFO, 2 = no INFO/WARN, 3 = only fatal
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # adjust if you want more/less logs [web:21][web:23]

# # Optional: disable oneDNN custom ops if you don't want that message
# # or if you care about strict numerical reproducibility across devices.
# os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # comment this out if you want oneDNN optimizations [web:7][web:29]


# ---- Your existing imports ----
import meridian
import arviz as az
import IPython

from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.analysis import formatter  # this is the documented location
from importlib.metadata import version, PackageNotFoundError 
import numpy as np
import pandas as pd
import inspect, sys
import pickle

# NumPy 1.x compatibility shim for Meridian 1.3.2
if not hasattr(np, 'concat'):
    np.concat = np.concatenate
    print("✅ Added np.concat compatibility shim")


from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

# ---- Resource checks ----
ram_gb = virtual_memory().total / 1e9
print("Your runtime has {:.1f} gigabytes of available RAM\n".format(ram_gb))

cpus = tf.config.list_physical_devices("CPU")
gpus = tf.config.list_physical_devices("GPU")  # modern API (no experimental) [web:26][web:28]

print("Num CPUs Available: ", len(cpus))
print("Num GPUs Available: ", len(gpus))

if gpus:
    print("GPU devices:")
    for gpu in gpus:
        print("  -", gpu)
else:
    print("No GPU detected by TensorFlow")

# ---- Simple GPU test (optional) ----
if gpus:
    try:
        with tf.device("/GPU:0"):
            a = tf.random.uniform((1000, 1000))
            b = tf.random.uniform((1000, 1000))
            c = tf.matmul(a, b)
        print("Simple matmul on GPU succeeded. Tensor shape:", c.shape)
    except Exception as e:
        print("GPU test failed, falling back to CPU. Error:", e)


print("✅ Libraries imported")
import warnings
warnings.filterwarnings('ignore')

print("✅ Meridian libraries imported")

Your runtime has 8.6 gigabytes of available RAM

Num CPUs Available:  1
Num GPUs Available:  0
No GPU detected by TensorFlow
✅ Libraries imported
✅ Meridian libraries imported


### 1. Load Files

In [6]:
# with open(model_path, 'rb') as f:
#     mmm = pickle.load(f)
# df_spend = pd.read_csv(data_path)


with open("meridian_model.pkl".replace('file:', ''), 'rb') as f:
    mmm = pickle.load(f)


df_spend = pd.read_csv("/Users/satishvavilapalli/Documents/application_demo/final_ds.csv")


In [7]:
channels = ['TV', 'MetaAds', 'TikTok', 'GoogleSearch', 'YouTube']
print(f"✅ Loaded: Model v1.3.2 | Data {df_spend.shape} | Channels {channels}")


✅ Loaded: Model v1.3.2 | Data (6240, 28) | Channels ['TV', 'MetaAds', 'TikTok', 'GoogleSearch', 'YouTube']


In [8]:
print("=" * 80)
print("🔍 RUNNING MODEL QUALITY CHECKS")
print("=" * 80)

reviewer.ModelReviewer(mmm).run()

🔍 RUNNING MODEL QUALITY CHECKS


2025-12-11 22:11:18.185797: I external/local_xla/xla/service/service.cc:163] XLA service 0x600003ecde00 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-12-11 22:11:18.185956: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1765471278.461875  288713 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2025-12-11 22:11:19.385596: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


Model Quality Checks
Overall Status: PASS
Summary: Passed: No major quality issues were identified.

Check Results:
----------------------------------------
Convergence Check:
  Status: PASS
  Recommendation: The model has likely converged, as all parameters have R-hat values < 1.2.
----------------------------------------
Baseline Check:
  Status: PASS
  Recommendation: The posterior probability that the baseline is negative is 0.00. We recommend visually inspecting the baseline time series in the Model Fit charts to confirm this.
----------------------------------------
BayesianPPP Check:
  Status: PASS
  Recommendation: The Bayesian posterior predictive p-value is 0.94. The observed total outcome is consistent with the model's posterior predictive distribution.
----------------------------------------
GoodnessOfFit Check:
  Status: PASS
  Recommendation: R-squared = 0.7737, MAPE = 0.2558, and wMAPE = 0.1998. These goodness-of-fit metrics are intended for guidance and relative compar

### 2. Model Diagnostics

In [10]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

alt.LayerChart(...)

In [11]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

alt.LayerChart(...)

### 3.Getting Results into DataFrame

#### 3.1 Incremental Outcome

In [15]:
# import meridian
# from meridian.analysis import visualizer, analyzer
# import inspect, sys

print("meridian version:", getattr(meridian, "__version__", "no __version__"))
print("visualizer path:", visualizer.__file__)
print("analyzer path:", inspect.getsourcefile(analyzer.Analyzer))

print("ModelFit init signature:", inspect.signature(visualizer.ModelFit.__init__))
print("Has expectedvsactualdata on Analyzer?",
      hasattr(analyzer.Analyzer, "expectedvsactualdata"))

print("meridian in sys.modules:", [k for k in sys.modules.keys() if k.startswith("meridian")])


meridian version: 1.4.0
visualizer path: /Users/satishvavilapalli/Documents/app_demo_v1/venv/lib/python3.12/site-packages/meridian/analysis/visualizer.py
analyzer path: /Users/satishvavilapalli/Documents/app_demo_v1/venv/lib/python3.12/site-packages/meridian/analysis/analyzer.py
ModelFit init signature: (self, meridian: meridian.model.model.Meridian, use_kpi: bool = False, confidence_level: float = 0.9)
Has expectedvsactualdata on Analyzer? False
meridian in sys.modules: ['meridian.backend.config', 'meridian.backend', 'meridian.constants', 'meridian.model.adstock_hill', 'meridian.data.arg_builder', 'meridian.data.time_coordinates', 'meridian.data.input_data', 'meridian.data.input_data_builder', 'meridian.data.data_frame_input_data_builder', 'meridian.data.load', 'meridian.data.nd_array_input_data_builder', 'meridian.data', 'meridian.model.knots', 'meridian.model.prior_distribution', 'meridian.model.spec', 'meridian.model.transformers', 'meridian.model.media', 'meridian.model.context', 

In [16]:
# List method names
methods = [name for name, obj in inspect.getmembers(analyzer.Analyzer)
           if inspect.isfunction(obj) or inspect.ismethoddescriptor(obj)]
methods


['__delattr__',
 '__dir__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__le__',
 '__lt__',
 '__ne__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '_calculate_baseline_expected_outcome',
 '_can_split_by_holdout_id',
 '_check_kpi_transformation',
 '_compute_cpik_aggregate',
 '_compute_effectiveness_aggregate',
 '_compute_pct_of_contribution',
 '_compute_roi_aggregate',
 '_compute_spend_data_aggregate',
 '_filter_holdout_id_for_selected_geos_and_times',
 '_get_adstock_dataframe',
 '_get_causal_param_names',
 '_get_channel_hill_histogram_dataframe',
 '_get_hill_curves_dataframe',
 '_get_hill_histogram_dataframe',
 '_get_incremental_kpi',
 '_get_kpi_means',
 '_get_scaled_data_tensors',
 '_get_transformed_media_and_beta',
 '_impute_and_aggregate_spend',
 '_incremental_outcome_impl',
 '_inverse_outcome',
 '_mean_and_ci_by_eval_set',
 '_predictive_accuracy_helper',
 '_use_kpi',


In [18]:
# getting the function signature
# On the class
print(hasattr(analyzer.Analyzer, "incremental_outcome"))

# Or on your instance
an = analyzer.Analyzer(mmm)
print(hasattr(an, "incremental_outcome"))

# If True, see the signature
if hasattr(an, "incremental_outcome"):
    print(inspect.signature(an.incremental_outcome))


True
True
(use_posterior: bool = True, new_data: meridian.analysis.analyzer.DataTensors | None = None, non_media_baseline_values: collections.abc.Sequence[float] | None = None, scaling_factor0: float = 0.0, scaling_factor1: float = 1.0, selected_geos: collections.abc.Sequence[str] | None = None, selected_times: collections.abc.Sequence[str] | collections.abc.Sequence[bool] | None = None, media_selected_times: collections.abc.Sequence[str] | collections.abc.Sequence[bool] | None = None, aggregate_geos: bool = True, aggregate_times: bool = True, inverse_transform_outcome: bool = True, use_kpi: bool = False, by_reach: bool = True, include_non_paid_channels: bool = True, batch_size: int = 100) -> tensorflow.python.framework.tensor.Tensor


In [21]:
n = analyzer.Analyzer(mmm)

incremental_kpi = an.incremental_outcome(
    use_posterior=True,
    new_data=None,
    non_media_baseline_values=None,
    scaling_factor0=0.0,          # baseline (media off)
    scaling_factor1=1.0,          # current media
    selected_geos=None,
    selected_times=None,
    media_selected_times=None,
    aggregate_geos=True,          # sum across geos
    aggregate_times=True,         # sum across time
    inverse_transform_outcome=True,
    use_kpi=True,                 # KPI instead of revenue
    by_reach=True,
    include_non_paid_channels=True,
    batch_size=100,
)

In [23]:
import numpy as np

inc = incremental_kpi.numpy()                 # shape (2, 300, 7)
inc_mean = inc.mean(axis=(0, 1))             # shape (7,)

channels = mmm.input_data.get_all_channels()  # or appropriate accessor
for ch, val in zip(channels, inc_mean):
    print(f"{ch}: {val:,.0f}")


TV: 2,975,393,536
Meta_Ads: 1,424,912,384
TikTok: 787,557,568
Google_Search: 3,068,728,064
YouTube: 3,374,706,944
Organic_TV: 599,527,040
Promo: 1,752,639,488


In [32]:

inc = an.incremental_outcome(
    use_posterior=True,
    new_data=None,
    non_media_baseline_values=None,
    scaling_factor0=0.0,
    scaling_factor1=1.0,
    selected_geos=None,
    selected_times=None,
    media_selected_times=None,
    aggregate_geos=True,      # national level
    aggregate_times=False,    # keep time dimension
    inverse_transform_outcome=True,
    use_kpi=True,
    by_reach=True,
    include_non_paid_channels=True,
    batch_size=100,
)  # shape: (chains, draws, times, channels) or similar [file:59][web:82]


inc_np = inc.numpy()
inc_mean = inc_np.mean(axis=(0, 1))   # time × channel

times = mmm.input_data.time.values
channels = mmm.input_data.get_all_channels()

df_inc = pd.DataFrame(inc_mean, columns=channels)
df_inc.insert(0, "time", times)

# convert selected columns to int
int_cols = ["TV", "Meta_Ads", "TikTok", "Google_Search", "YouTube", "Organic_TV", "Promo"]
df_inc[int_cols] = df_inc[int_cols].astype(int)

In [33]:
df_inc.head()

,time,TV,Meta_Ads,TikTok,Google_Search,YouTube,Organic_TV,Promo
0,2021-01-25,7996865,2017379,1600063,11855206,9314200,1933496,5853062
1,2021-02-01,12559904,6081485,2785410,16624770,11569547,3176501,3525287
2,2021-02-08,14174749,6967991,2957468,17801966,12425405,2766748,13820922
3,2021-02-15,15837121,6724768,2685425,18485440,15088945,2675208,8160245
4,2021-02-22,17756274,7587558,4283058,19767322,18124614,2480018,7958745


### 3.2 Expected Outcome

In [34]:

print(inspect.signature(an._calculate_baseline_expected_outcome))
print(inspect.getdoc(an._calculate_baseline_expected_outcome))


(non_media_baseline_values: collections.abc.Sequence[float] | None = None, **expected_outcome_kwargs) -> tensorflow.python.framework.tensor.Tensor
Calculates either the posterior or prior expected outcome of baseline.

This is a wrapper for expected_outcome() that automatically sets the
following argument values:
  1) `new_media` is set to all zeros
  2) `new_reach` is set to all zeros
  3) `new_organic_media` is set to all zeros
  4) `new_organic_reach` is set to all zeros
  5) `new_non_media_treatments` is set to the counterfactual values
  according to the `non_media_baseline_values` argument
  6) `new_controls` are set to historical values

All other arguments of `expected_outcome` can be passed to this method.

Args:
  non_media_baseline_values: Optional list of shape
    `(n_non_media_channels,)`. Each element is a float denoting a fixed
    value that will be used as the baseline for the given channel. It is
    expected that they are scaled by population for the channels where


In [35]:
# Baseline time series over geo × time (posterior mean & CI)
baseline_ds = an.expected_vs_actual_data(
    aggregate_geos=False,
    aggregate_times=False,
    use_kpi=True,             # or False
    split_by_holdout_id=False,
    non_media_baseline_values=None,
    confidence_level=0.9,
)

# Baseline summary metrics (national)
baseline_summary = an.baseline_summary_metrics(
    aggregate_times=True,
    use_kpi=True,
    confidence_level=0.9,
)

In [36]:
baseline_summary

<xarray.Dataset> Size: 264B
Dimensions:              (metric: 4, distribution: 2)
Coordinates:
  * metric               (metric) <U6 96B 'mean' 'median' 'ci_lo' 'ci_hi'
  * distribution         (distribution) <U9 72B 'prior' 'posterior'
    channel              <U8 32B 'baseline'
Data variables:
    baseline_outcome     (metric, distribution) float32 32B 6.696e+10 ... 5.5...
    pct_of_contribution  (metric, distribution) float32 32B 76.22 78.8 ... 84.36

In [37]:
baseline_ds

<xarray.Dataset> Size: 182kB
Dimensions:   (geo: 40, time: 156, metric: 3)
Coordinates:
  * geo       (geo) <U5 800B 'Geo0' 'Geo1' 'Geo2' ... 'Geo37' 'Geo38' 'Geo39'
  * time      (time) <U10 6kB '2021-01-25' '2021-02-01' ... '2024-01-15'
  * metric    (metric) <U5 60B 'mean' 'ci_lo' 'ci_hi'
Data variables:
    expected  (geo, time, metric) float32 75kB 2.283e+06 2.004e+06 ... 2.36e+07
    baseline  (geo, time, metric) float32 75kB 2.094e+06 1.63e+06 ... 1.907e+07
    actual    (geo, time) float32 25kB 1.955e+06 2.064e+06 ... 2.098e+07
Attributes:
    confidence_level:  0.9

In [68]:
ds_ex_vs_ac = an.expected_vs_actual_data(
    aggregate_geos=False,
    aggregate_times=False,
    use_kpi=True,
    split_by_holdout_id=False,
    non_media_baseline_values=None,
    confidence_level=0.9,
)

print(ds_ex_vs_ac)                # summary, coords and variables
print(ds_ex_vs_ac['baseline'])    # baseline stats
print(ds_ex_vs_ac['baseline'].sel(geo=ds_ex_vs_ac.geo.values[0]).values[:5])


<xarray.Dataset> Size: 182kB
Dimensions:   (geo: 40, time: 156, metric: 3)
Coordinates:
  * geo       (geo) <U5 800B 'Geo0' 'Geo1' 'Geo2' ... 'Geo37' 'Geo38' 'Geo39'
  * time      (time) <U10 6kB '2021-01-25' '2021-02-01' ... '2024-01-15'
  * metric    (metric) <U5 60B 'mean' 'ci_lo' 'ci_hi'
Data variables:
    expected  (geo, time, metric) float32 75kB 2.283e+06 2.004e+06 ... 2.36e+07
    baseline  (geo, time, metric) float32 75kB 2.094e+06 1.63e+06 ... 1.907e+07
    actual    (geo, time) float32 25kB 1.955e+06 2.064e+06 ... 2.098e+07
Attributes:
    confidence_level:  0.9
<xarray.DataArray 'baseline' (geo: 40, time: 156, metric: 3)> Size: 75kB
array([[[2.0938835e+06, 1.6299689e+06, 2.4014502e+06],
        [1.3552958e+06, 8.7126319e+05, 1.7135628e+06],
        [1.9092839e+06, 1.4723181e+06, 2.2410742e+06],
        ...,
        [2.2185238e+06, 1.7887626e+06, 2.5348648e+06],
        [2.1688028e+06, 1.7344030e+06, 2.4981638e+06],
        [2.2596192e+06, 1.8200758e+06, 2.5936035e+06]],

 

In [61]:
# Extract mean baseline for one geo
g0 = ds_ex_vs_ac['baseline'].sel(geo='Geo0')          # (time, metric)
g0_mean = g0.sel(metric='mean')              # (time,)

df_baseline_mean = g0_mean.to_series().head(5).reset_index()
df_baseline_mean.columns = ['time', 'baseline_mean']
df_baseline_mean['baseline_mean'] = df_baseline_mean['baseline_mean'].map(lambda x: f"{x:,.0f}")

df_baseline_mean.head()


,time,baseline_mean
0,2021-01-25,"2,093,884"
1,2021-02-01,"1,355,296"
2,2021-02-08,"1,909,284"
3,2021-02-15,"2,028,286"
4,2021-02-22,"1,674,975"


In [62]:
# extracting National baseline 
national_baseline = ds['baseline'].sel(metric='mean').sum(dim='geo')  # (time,)
nat_df = national_baseline.to_series().reset_index()
nat_df.columns = ['time', 'baseline_mean_nat']
nat_df['baseline_mean_nat'] = nat_df['baseline_mean_nat'].map(lambda x: f"{x/1e6:.2f} M")

nat_df.head()


,time,baseline_mean_nat
0,2021-01-25,318.31 M
1,2021-02-01,303.23 M
2,2021-02-08,385.51 M
3,2021-02-15,341.14 M
4,2021-02-22,291.45 M


In [63]:
# Expected outcome (national, by time)
exp_outcome = an.expected_outcome(
    use_posterior=True,
    selected_geos=None,
    selected_times=None,
    aggregate_geos=True,
    aggregate_times=False,
    inverse_transform_outcome=True,
    use_kpi=True,
    batch_size=100,
)

exp_np = exp_outcome.numpy()
exp_mean = exp_np.mean(axis=(0, 1))   # time
df_exp = pd.DataFrame({"time": times, "expected": exp_mean})

# Merge and compute baseline vs channels
df_expected_outcome = df_inc.merge(df_exp, on="time")
df_expected_outcome["channels_total"] = df_expected_outcome[channels].sum(axis=1)
df_expected_outcome["baseline"] = df_expected_outcome["expected"] - df_expected_outcome["channels_total"]

cols_to_int = [
    "TV", "Meta_Ads", "TikTok", "Google_Search", "YouTube",
    "Organic_TV", "Promo", "expected", "channels_total", "baseline"
]

df_expected_outcome[cols_to_int] = df_expected_outcome[cols_to_int].astype(int)

In [64]:
df_expected_outcome.head()

,time,TV,Meta_Ads,TikTok,Google_Search,YouTube,Organic_TV,Promo,expected,channels_total,baseline
0,2021-01-25,7996865,2017379,1600063,11855206,9314200,1933496,5853062,358884544,40570271,318314273
1,2021-02-01,12559904,6081485,2785410,16624770,11569547,3176501,3525287,359548448,56322904,303225544
2,2021-02-08,14174749,6967991,2957468,17801966,12425405,2766748,13820922,456424768,70915249,385509519
3,2021-02-15,15837121,6724768,2685425,18485440,15088945,2675208,8160245,410795168,69657152,341138016
4,2021-02-22,17756274,7587558,4283058,19767322,18124614,2480018,7958745,369409248,77957589,291451659


In [65]:
last = df_expected_outcome.iloc[-1]
row = {ch: last[ch] for ch in channels}
row["baseline"] = last["baseline"]
pretty = {k: f"{v/1e6:.2f} M" for k, v in row.items()}
print(pretty)


{'TV': '20.19 M', 'Meta_Ads': '9.65 M', 'TikTok': '4.91 M', 'Google_Search': '20.68 M', 'YouTube': '22.04 M', 'Organic_TV': '4.30 M', 'Promo': '7.71 M', 'baseline': '338.53 M'}


### 3.3 Expected VS Actual

In [67]:
import pandas as pd

g = "Geo0"

df_expected_vs_actual = pd.DataFrame({
    "time": ds_ex_vs_ac.time.values,
    "expected_mean": ds_ex_vs_ac["expected"].sel(geo=g, metric="mean").values,
    "baseline_mean": ds_ex_vs_ac["baseline"].sel(geo=g, metric="mean").values,
    "actual": ds_ex_vs_ac["actual"].sel(geo=g).values,
})

# Nice formatting
for col in ["expected_mean", "baseline_mean", "actual"]:
    df_expected_vs_actual[col] = df_expected_vs_actual[col].map(lambda x: f"{x:,.0f}")

print(df_expected_vs_actual.head(5))


         time expected_mean baseline_mean     actual
0  2021-01-25     2,282,658     2,093,884  1,954,577
1  2021-02-01     1,711,103     1,355,296  2,064,250
2  2021-02-08     2,407,720     1,909,284  2,086,383
3  2021-02-15     2,625,666     2,028,286  2,826,432
4  2021-02-22     2,141,315     1,674,975  3,551,929


In [69]:
# National level

nat = ds_ex_vs_ac.sum(dim="geo")  # aggregate over geo

nat_df = pd.DataFrame({
    "time": nat.time.values,
    "expected_mean": nat["expected"].sel(metric="mean").values,
    "baseline_mean": nat["baseline"].sel(metric="mean").values,
    "actual": nat["actual"].values,
})


In [71]:
nat_df.head()

,time,expected_mean,baseline_mean,actual
0,2021-01-25,358884640.0,318314432.0,386300928.0
1,2021-02-01,359548448.0,303225568.0,355303968.0
2,2021-02-08,456424864.0,385509568.0,438308800.0
3,2021-02-15,410795168.0,341138048.0,418612384.0
4,2021-02-22,369409152.0,291451584.0,355785376.0


### 3.4 ROI & mROI

In [79]:
# ROI


roi_tensor = an.roi(
    use_posterior=True,
    new_data=None,
    selected_geos=None,
    selected_times=None,
    aggregate_geos=True,
    use_kpi=True,          # or False for revenue
    batch_size=100,
)


roi_np = roi_tensor.numpy()
roi_mean = roi_np.mean(axis=(0, 1))   # per channel
channels = mmm.input_data.get_all_channels()
# for ch, v in zip(channels, roi_mean):
#     print(f"{ch}: {v:.2f}")


df_roi = pd.DataFrame(
    list(zip(channels, roi_mean)),
    columns=["channel", "roi_mean"]
)

df_roi

,channel,roi_mean
0,TV,73.463371
1,Meta_Ads,45.494839
2,TikTok,65.198723
3,Google_Search,34.927376
4,YouTube,70.703011


In [81]:
# mROI

mroi_tensor = an.marginal_roi(
    incremental_increase=0.01,   # 1% bump in spend
    use_posterior=True,
    new_data=None,
    selected_geos=None,
    selected_times=None,
    aggregate_geos=True,
    by_reach=True,
    use_kpi=True,                # or False for revenue
    batch_size=100,
)


mroi_np = mroi_tensor.numpy()
mroi_mean = mroi_np.mean(axis=(0, 1))
# for ch, v in zip(channels, mroi_mean):
#     print(f"{ch}: {v:.2f}")


df_mroi = pd.DataFrame(
    list(zip(channels, mroi_mean)),
    columns=["channel", "mroi_mean"]
)

df_mroi

,channel,mroi_mean
0,TV,36.589035
1,Meta_Ads,23.413635
2,TikTok,36.740582
3,Google_Search,17.720299
4,YouTube,33.471569


### 3.5 Response Curves

In [82]:
from meridian import constants as c

an = analyzer.Analyzer(mmm)

rc_ds = an.response_curves(
    confidence_level=0.9,
    selected_times=None,     # or list of dates
    by_reach=True,           # RF channels: curve by reach (True) or frequency (False)
    use_kpi=True,            # True = KPI, False = revenue
)


In [83]:
rc_ds

<xarray.Dataset> Size: 2kB
Dimensions:              (spend_multiplier: 11, channel: 5, metric: 3)
Coordinates:
  * spend_multiplier     (spend_multiplier) float64 88B 0.0 0.2 0.4 ... 1.8 2.0
  * channel              (channel) object 40B 'TV' 'Meta_Ads' ... 'YouTube'
  * metric               (metric) <U5 60B 'mean' 'ci_lo' 'ci_hi'
Data variables:
    spend                (spend_multiplier, channel) float32 220B 0.0 ... 9.5...
    incremental_outcome  (spend_multiplier, channel, metric) float64 1kB 0.0 ...
Attributes:
    confidence_level:  0.9

In [91]:

df_rc = rc_ds[constants.INCREMENTAL_OUTCOME].to_dataframe().reset_index()
df_rc_mean = df_rc[df_rc["metric"] == constants.MEAN].drop(columns=["metric"])


df_rc_mean["incremental_outcome"] = df_rc_mean[constants.INCREMENTAL_OUTCOME].apply(lambda x: f"{int(x):,}")


df_rc_mean.tail()

,spend_multiplier,channel,incremental_outcome
150,2.0,TV,"4,008,555,264"
153,2.0,Meta_Ads,"1,948,875,904"
156,2.0,TikTok,"1,120,807,296"
159,2.0,Google_Search,"4,156,393,216"
162,2.0,YouTube,"4,480,370,176"
